In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login
import torch
import pandas as pd
from tqdm import tqdm
import re
import pandas as pd

2025-05-22 15:49:45.791872: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747928986.070314      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747928986.145101      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
from huggingface_hub import login

login("KEY") #### INSERT KEY HERE

In [ ]:
ds = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:50]") 

model_id = "mistralai/Mistral-7B-Instruct-v0.2"


#ds = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:1%]")

# Load model (adjust model_id if needed)
#model_id = "meta-llama/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

llm = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=50)

# Prompt builder
def build_prompt(question, answer, documents):
    docs_text = "\n".join([f"{i+1}. {doc}" for i, doc in enumerate(documents)])
    prompt = f"""You are given a question and its ground truth answer. Below are 10 documents retrieved by a system.

Your task is to assign a relevance score from 0 to 5 to each document based on how useful it is in answering the question.

Respond as a numbered list like:
1. 3
2. 0
...

### Question:
{question}

### Answer:
{answer}

### Documents:
{docs_text}
"""
    return prompt

# Parse the output
def parse_scores(text):
    scores = {}
    matches = re.findall(r"(\d+)[\.\:\)]\s*([0-5])", text)
    for doc_id, score in matches:
        scores[int(doc_id)-1] = int(score)
    return [scores.get(i, 0) for i in range(10)]  # default 0 if missing

# Run inference per QA pair
all_results = []

for item in tqdm(ds, desc="Scoring QA pairs"):
    question = item["question"]
    answer = item["answer"]
    docs = item["documents"]
    
    prompt = build_prompt(question, answer, docs)

   # print(f"------\n{prompt}\n-------")
    
    with torch.inference_mode():
        output = llm(prompt)[0]['generated_text']
    scores = parse_scores(output)

    for i, score in enumerate(scores):
        all_results.append({
            "question": question,
            "answer": answer,
            "doc_id": i,
            "document": docs[i],
            "relevance_score": score
        })

# Save to DataFrame
df = pd.DataFrame(all_results)
df["rank"] = df.groupby("question")["relevance_score"].rank(method="dense", ascending=False)

In [56]:
s= "\n\n".join(ds["documents"][0])
q = ds["question"][0]
a = ds["answer"][0]

print(f"QUESTION: {q}\n")
print(f"ANSWER: {a}\n")
print(f"CONTEXT: \n{s}")

QUESTION: Describe the cultural impact and legacy of the 1967 Disney film 'The Jungle Book' on animation and popular culture.

ANSWER: The 1967 Disney film 'The Jungle Book' has had a profound impact on animation and popular culture. It was the last animated film produced by Walt Disney himself, marking an end to an era of his personal touch in Disney films. The movie is notable for using familiar celebrity voices to shape character personalities, such as Phil Harris improvising lines for Baloo, a rare practice in Disney movies of the time which later became a more common trend in animation. This practice helped create more relatable and memorable characters, contributing to the film's lasting popularity. Culturally, 'The Jungle Book' introduced audiences to a blend of humor, adventure, and catchy music, which have been influential in defining the modern animated musical genre. The film's success led to the expansion of 'The Jungle Book' into a media franchise, which includes sequels a

In [19]:
df[0:10]

,question,answer,doc_id,document,relevance_score,rank
0,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,0,decided to make the story more straightforward...,2,1.0
1,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,1,"and settings. In 2016, a Baloo figure was rele...",0,2.0
2,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,2,The Jungle Book (franchise) The Jungle Book is...,0,2.0
3,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,3,and follows her into the Man-Village. After Mo...,0,2.0
4,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,4,creature. So we took some of the distinctive W...,0,2.0
5,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,5,"each other. ""The Jungle Book"" also marks the l...",0,2.0
6,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,6,"one merit: If you have unruly children, it may...",0,2.0
7,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,7,"the King of Thieves"". In December 2010, a piec...",0,2.0
8,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,8,"Pooh and the Honey Tree"" (1966) and ""Winnie th...",0,2.0
9,Describe the cultural impact and legacy of the...,The 1967 Disney film 'The Jungle Book' has had...,9,its Indian audiences as the book and the Disne...,0,2.0


In [ ]:
roups = [df.iloc[i:i+10] for i in range(0, len(df), 10)]

result = []
for group in groups:
    tuples = [(row['relevance_score'], row['doc_id']) for _, row in group.iterrows()]
    sorted_tuples = sorted(tuples, key=lambda x: x[0], reverse=True)
    result.append(sorted_tuples)

In [ ]:
np.save(f"/kaggle/working/LLM-teacher_Mistral-7B-Instruct-v0.2.npy", result)